### HIPPOCORPUS analysis and comparison

#### Score HIPPOCORPUS data and plot

In [ ]:
import os, json, time
import pandas as pd
import openai
from scipy.stats import f_oneway, ttest_rel, ttest_ind
import matplotlib.pyplot as plt

k = 'x'# ADD OPENAI API KEY HERE
client = openai.OpenAI(api_key=k)

SYSTEM_PROMPT = """Your task is score text on three metrics: how concrete (vs abstract) it is, how rich in detail it is, and how specific (vs general) it is.

Return ONLY a JSON dictionary with 3 keys, each a float 0-1:

{
  "concrete_vs_abstract": 0-1,
  "rich_vs_poor_details": 0-1,
  "specific_vs_general":  0-1
}

A higher score corresponds to more concrete, richer in detail, or more specific text."""

def llm_scores(text: str,
               model: str = "gpt-4o-mini",
               max_retries: int = 5) -> dict:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text[:16_000]}
    ]
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=msgs,
                temperature=0.0,
                response_format={"type": "json_object"},
            )
            return json.loads(resp.choices[0].message.content)
        except openai.RateLimitError:
            time.sleep(2 + attempt)          # back-off then retry
        except openai.OpenAIError as e:
            print("OpenAI error:", e)
            time.sleep(3)
    # if still failing:
    return {"concrete_vs_abstract": None,
            "rich_vs_poor_details": None,
            "specific_vs_general":  None}



In [ ]:
df = pd.read_csv("hippoCorpusV2.csv")
stories = (
    df[["recAgnPairId", "memType", "story"]]
      .dropna(subset=["story"])
      .reset_index(drop=True)
)

# records = []
# for row in stories.itertuples(index=False):
#     js = llm_scores(row.story)
#     js.update({"recAgnPairId": row.recAgnPairId,
#                "memType":      row.memType})
#     records.append(js)

# score_df = pd.DataFrame(records)
# score_df.to_csv("story_llm_ratings.csv", index=False)
# print("Saved → story_llm_ratings.csv")

In [ ]:
# Load the scores produced earlier
df = pd.read_csv("story_llm_ratings.csv")

metrics = [
    ("concrete_vs_abstract", "Concreteness"),
    ("rich_vs_poor_details", "Richness in detail"),
    ("specific_vs_general",  "Specificity")
]

# Only plot recalled vs retold
groups = ["recalled", "retold"]
colors = ["#4C78A8", "#F58518"]

fig, axes = plt.subplots(1, 3, figsize=(6, 3), sharey=True)

# --- compute global y-limits across all metrics ---
all_vals = []
for col, _ in metrics:
    means = df.groupby("memType")[col].mean().reindex(groups)
    sems  = df.groupby("memType")[col].sem().reindex(groups)
    all_vals.append((means - sems).min())
    all_vals.append((means + sems).max())

y_min_data = min(all_vals)
y_max_data = max(all_vals)

low_pad  = 0.1 * (y_max_data - y_min_data)
base_sig = y_max_data + 0.06
y_top    = base_sig + 0.02   # only one pair

for i, (col, title) in enumerate(metrics):
    ax = axes[i]
    ax.set_ylim(0.45,1) #y_min_data - low_pad, y_top)

    # plot bars -------------------------------------------------------
    means = df.groupby("memType")[col].mean().reindex(groups)
    sems  = df.groupby("memType")[col].sem().reindex(groups)
    ax.bar(groups, means, yerr=sems, color=colors, capsize=4, alpha=.9)
    ax.set_title(title)
    if i == 0:
        ax.set_ylabel("Score")
    ax.set_xticklabels(groups, rotation=20)

    # significance line for recalled vs retold ------------------------
    g1, g2 = "recalled", "retold"
    x1, x2 = groups.index(g1), groups.index(g2)
    y  = base_sig
    g1v = df[df.memType==g1][col].dropna()
    g2v = df[df.memType==g2][col].dropna()
    t, p = ttest_ind(g1v, g2v, equal_var=False)

    sym = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else "ns"
    ax.plot([x1, x1, x2, x2], [y, y+0.01, y+0.01, y], lw=1.2, c="k")
    ax.text((x1+x2)/2, y+0.003, sym, ha="center", va="bottom")

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("twobars.png")


In [ ]:
import sys, importlib, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import sem
from pathlib import Path
from tqdm import tqdm
import statsmodels.api as sm_api
import statsmodels.formula.api as smf

# Drop rows with missing keys
df = df.dropna(subset=["recAgnPairId","memType"])
stories = stories.dropna(subset=["recAgnPairId","memType"])

# Now check
assert not df[["recAgnPairId","memType"]].isna().any().any(), "Keys have NaNs"
assert not stories[["recAgnPairId","memType"]].isna().any().any(), "Keys have NaNs"

# Deduplicate on the join keys
dfu = df.drop_duplicates(["recAgnPairId","memType"])
stu = stories.drop_duplicates(["recAgnPairId","memType"])

# Safe one-to-one merge
merged = dfu.merge(stu,
                   on=["recAgnPairId","memType"],
                   how="left",
                   validate="one_to_one")

merged = df

df = df.merge(stories, on=["recAgnPairId","memType"], how="left")
df["story"] = df["story"].fillna("")
df["word_count"] = df["story"].str.count(r"\S+")

print(f"Merged dataframe shape: {df.shape}")
print(df[["memType","word_count"]].groupby("memType").describe(percentiles=[]))

# --- 1) ANCOVA for each metric (memType + word_count covariate) ----------------
print("\n=== ANCOVA (metric ~ memType + word_count) ===")
ancova_results = {}
for col, title in metrics:
    # Drop NAs
    sub = df[["memType", "word_count", col]].dropna()
    model = smf.ols(f"{col} ~ C(memType) + word_count", data=sub).fit()
    anova_table = sm_api.stats.anova_lm(model, typ=2)
    ancova_results[col] = (model, anova_table)
    print(f"\n[{title}]")
    display(anova_table)

# --- 2) Length-matched reanalysis via decile stratified sampling ----------------
def length_matched_df(df, length_col="word_count", group_col="memType", n_bins=10, seed=0):
    rng = np.random.default_rng(seed)
    d = df.copy()
    # bin by deciles on the whole corpus
    d["_bin"] = pd.qcut(d[length_col].rank(method="first"), q=n_bins, labels=False, duplicates="drop")
    strata = []
    for _, g in d.groupby("_bin"):
        # equalize counts across memType within this bin
        counts = g[group_col].value_counts()
        if len(counts) < len(groups):
            continue  # skip bins missing a group
        m = counts.min()
        if m == 0:
            continue
        sampled = []
        for mem in groups:
            pool = g[g[group_col] == mem]
            if len(pool) >= m:
                sampled.append(pool.sample(m, random_state=int(rng.integers(1e9))))
        if sampled:
            strata.append(pd.concat(sampled, axis=0))
    out = pd.concat(strata, axis=0).drop(columns=["_bin"])
    return out

matched = length_matched_df(df)
print("\nLength-matched subset sizes:")
print(matched["memType"].value_counts())


#### Plot data from simulations

In [ ]:
ratings_df = pd.read_csv("output/data/story_llm_ratings_simulated.csv")

metrics = [
    ("concrete_vs_abstract", "Concreteness"),
    ("rich_vs_poor_details", "Richness in detail"),
    ("specific_vs_general",  "Specificity"),
]

# Three versions to compare
groups = ["original", "encoded", "consolidated"]
colors = ["tomato", "#4C78A8", "#F58518"]

fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=True)

# ---- global y-limits & spacing (same style as your NFRD script) ---
all_means, all_sems = [], []
for col, _ in metrics:
    means = ratings_df.groupby("version")[col].mean().reindex(groups)
    sems  = ratings_df.groupby("version")[col].sem().reindex(groups)
    all_means.extend(means.values)
    all_sems.extend(sems.values)

y_min_data = float(min(m - s for m, s in zip(all_means, all_sems)))
y_max_data = float(max(m + s for m, s in zip(all_means, all_sems)))
data_range = max(0.01, y_max_data - y_min_data)

low_pad        = 0.05 * data_range
sig_gap        = 0.35  * data_range
text_gap       = 0.015 * data_range
base_sig       = y_max_data + sig_gap
extra_headroom = 0.9  * data_range
y_top          = base_sig + text_gap + extra_headroom

for ax in axes:
    ax.set_ylim(0.45, 1) #ax.set_ylim(y_min_data - low_pad, y_top)

def draw_bracket(ax, x1, x2, y_base, label, data_range):
    """NFRD-style significance bracket with clamped label position and tails."""
    margin       = 0.02  * data_range
    text_gap_loc = 1.0   * text_gap
    bracket_h    = 0.40  * text_gap
    tail_h       = 2.0   * text_gap

    y_text        = min(y_base + text_gap_loc, y_top - margin)
    y_bracket_top = min(y_base + bracket_h + tail_h, y_text - 0.3 * text_gap)
    y_bracket     = y_bracket_top - tail_h

    # horizontal
    ax.plot([x1, x1, x2, x2],
            [y_bracket, y_bracket + tail_h, y_bracket + tail_h, y_bracket],
            lw=1.2, c="k")
    # little downward tails
    ax.plot([x1, x1], [y_bracket, y_bracket - tail_h], lw=1.2, c="k")
    ax.plot([x2, x2], [y_bracket, y_bracket - tail_h], lw=1.2, c="k")

    ax.text((x1 + x2) / 2, y_text, label, ha="center", va="bottom")

from scipy.stats import ttest_rel, ttest_ind  # ensure imported

comparisons = [("original", "encoded"),
               ("original", "consolidated"),
               ("encoded",  "consolidated")]

for i, (ax, (col, title)) in enumerate(zip(axes, metrics)):
    means = ratings_df.groupby("version")[col].mean().reindex(groups)
    sems  = ratings_df.groupby("version")[col].sem().reindex(groups)

    ax.bar(groups, means, yerr=sems, color=colors, capsize=4, alpha=.9)
    ax.set_title(title)
    ax.tick_params(axis="x", labelrotation=20)
    if i == 0:
        ax.set_ylabel("Score")

    # pairwise brackets in the same style, stacked upward
    for j, (g1, g2) in enumerate(comparisons):
        x1, x2 = groups.index(g1), groups.index(g2)
        g1v = ratings_df.loc[ratings_df.version == g1, col].dropna()
        g2v = ratings_df.loc[ratings_df.version == g2, col].dropna()

        # use paired if you truly have within-item pairing; else Welch
        try:
            t, p = ttest_rel(g1v, g2v, nan_policy="omit")
            if pd.isna(t) or pd.isna(p):
                raise ValueError
        except Exception:
            t, p = ttest_ind(g1v, g2v, equal_var=False)

        sym = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"
        y_this = base_sig + j * (0.9 * sig_gap)  # slight compaction vs NFRD default
        draw_bracket(ax, x1, x2, y_this, sym, data_range)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("llm_versions_analysis.png", dpi=300, bbox_inches="tight")
plt.show()
